In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image

BASE_PATH = os.path.join("/kaggle", "input", "cassava-leaf-disease-classification")
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train_images")
TEST_IMG_DIR = os.path.join(BASE_PATH, "test_images")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TEST_CSV = os.path.join(BASE_PATH, "sample_submission.csv")  # contains correct ordering

assert os.path.isdir(TRAIN_IMG_DIR), f"Training image folder not found: {TRAIN_IMG_DIR}"
assert os.path.isdir(TEST_IMG_DIR), f"Test image folder not found: {TEST_IMG_DIR}"
assert os.path.isfile(TRAIN_CSV), f"Training CSV not found: {TRAIN_CSV}"
assert os.path.isfile(TEST_CSV), f"Test CSV not found: {TEST_CSV}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [2]:
class CassavaDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image_id"]
        label = int(self.df.iloc[idx]["label"])
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


data_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)



In [3]:
# Optimize data loading: use multiple workers and keep them alive across epochs.
# This change does not affect model architecture or training logic.
num_workers = max(
    1, os.cpu_count() // 2
)  # reasonable parallelism without oversubscription

full_dataset = CassavaDataset(
    csv_path=TRAIN_CSV, img_dir=TRAIN_IMG_DIR, transform=data_transform
)

val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 5)  # 5 disease classes
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)


def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            lbls = lbls.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
    return correct / total if total > 0 else 0.0


epochs = 8  # extended training for better convergence
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs = imgs.to(device)
        lbls = lbls.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / train_size
    val_acc = evaluate(val_loader)
    print(f"Epoch {epoch}/{epochs} - Loss: {epoch_loss:.4f} - Val Acc: {val_acc:.4f}")

    scheduler.step()  # update learning rate



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResN

Epoch 1/8 - Loss: 0.7310 - Val Acc: 0.6116
Epoch 2/8 - Loss: 0.5757 - Val Acc: 0.7308
Epoch 3/8 - Loss: 0.5299 - Val Acc: 0.7858
Epoch 4/8 - Loss: 0.4360 - Val Acc: 0.8312
Epoch 5/8 - Loss: 0.4058 - Val Acc: 0.7676
Epoch 6/8 - Loss: 0.3880 - Val Acc: 0.8264
Epoch 7/8 - Loss: 0.3222 - Val Acc: 0.8200
Epoch 8/8 - Loss: 0.2926 - Val Acc: 0.8259


In [4]:
test_df = pd.read_csv(TEST_CSV)  # contains image_id column in required order
test_images = test_df["image_id"].tolist()


class TestDataset(Dataset):
    def __init__(self, img_names, img_dir, transform=None):
        self.img_names = img_names
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name


# Use the same fast DataLoader settings for inference.
test_dataset = TestDataset(
    img_names=test_images, img_dir=TEST_IMG_DIR, transform=data_transform
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
)

model.eval()
predictions = []

with torch.no_grad():
    for imgs, names in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        preds = preds.cpu().numpy()
        predictions.extend(zip(names, preds))

submission_path = "./submission.csv"
with open(submission_path, "w") as f:
    f.write("image_id,label\n")
    for img_id, pred in predictions:
        f.write(f"{img_id},{int(pred)}\n")

print("Submission written to", submission_path)
print("First few lines:")
with open(submission_path, "r") as f:
    for _ in range(5):
        print(f.readline().strip())

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Submission written to ./submission.csv
First few lines:
image_id,label
1234294272.jpg,3
1234332763.jpg,3
1234375577.jpg,1
1234555380.jpg,1
